# Pydantic model to structure output

create an agent to simulate IT employees

In [ ]:
from pydantic_ai import Agent
from dotenv import load_dotenv 

load_dotenv()

employee_simulator_agent = Agent("openrouter:liquid/lfm-2.5-1.2b-thinking:free",
system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.

Fields to include in output:
- name
- age
- gender
- job_title
- salary in SEK per month
""")

result = await employee_simulator_agent.run("Simulate two employees")




In [2]:
print(result.output)

Here are two simulated IT employees in Sweden:

---

**Employee 1**  
- **Name:** Anna Schwabenstein  
- **Age:** 32  
- **Gender:** Female  
- **Job Title:** Software Engineer (AI Systems)  
- **Salary:** 58,500 SEK/month  

**Employee 2**  
- **Name:** Jens Nilsson  
- **Age:** 28  
- **Gender:** Male  
- **Job Title:** Data Analyst (Machine Learning)  
- **Salary:** 62,200 SEK/month  

--- 

Both employees reflect diverse gender representation and roles typical in Sweden’s tech sector. Let me know if you’d like adjustments!


In [3]:
with open("simulated_employees.md", "w") as file:
    file.write(result.output)

## Get more structured output

issue with above:
- output structure vary 
- hard to work with the data e.g. compute mean of salaries

want: 
- repeatable structure 

In [3]:
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent


class EmployeeModel(BaseModel):
    name: str = Field(
        description="Mostly swedish names, but could be foreign names as well"
    )
    age: int = Field(description="age should be between 18 and 67")
    gender: Literal["Male", "Female"]
    experience_level: Literal["Entry", "Mid level", "Senior", "Expert"]
    job_title: str
    salary: int = Field(
        gte=30_000,
        lte=50_000,
        description="salary should be between 30k and 50k, the higher experience level, the higher salary",
    )


employee_simulator_agent = Agent(
    "openrouter:openai/gpt-oss-20b:free",
    system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""",
)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=EmployeeModel)
result

/var/folders/tj/m_tpmk9937l71skgpnzqylbm0000gn/T/ipykernel_94249/1484438454.py:14: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(


AgentRunResult(output=EmployeeModel(name='Anna Karlsson', age=30, gender='Female', experience_level='Mid level', job_title='Data Engineer', salary=38000))

In [5]:
result.output

EmployeeModel(name='Anna Karlsson', age=30, gender='Female', experience_level='Mid level', job_title='Data Engineer', salary=38000)

In [7]:
result.output.salary+5000

43000

In [15]:
employee_simulator_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""", retries=1
)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=list[EmployeeModel], )
result

AgentRunResult(output=[EmployeeModel(name='Anna Eriksson', age=28, gender='Female', experience_level='Entry', job_title='Data Scientist', salary=35000), EmployeeModel(name='Raj Patel', age=35, gender='Male', experience_level='Mid level', job_title='Data Engineer', salary=40000), EmployeeModel(name='Sofia Martinez', age=42, gender='Female', experience_level='Senior', job_title='AI Engineer', salary=48000)])

In [19]:
result.output[0]

EmployeeModel(name='Anna Eriksson', age=28, gender='Female', experience_level='Entry', job_title='Data Scientist', salary=35000)

In [ ]:
# BaseModel -> dictionary
result.output[0].model_dump()

{'name': 'Anna Eriksson',
 'age': 28,
 'gender': 'Female',
 'experience_level': 'Entry',
 'job_title': 'Data Scientist',
 'salary': 35000}

TODO: 
- result.output make into list of dictionaries
- create pandas dataframe based on this list
- export a csv file of our simulated employees

In [22]:
employees = result.output
employees

[EmployeeModel(name='Anna Eriksson', age=28, gender='Female', experience_level='Entry', job_title='Data Scientist', salary=35000),
 EmployeeModel(name='Raj Patel', age=35, gender='Male', experience_level='Mid level', job_title='Data Engineer', salary=40000),
 EmployeeModel(name='Sofia Martinez', age=42, gender='Female', experience_level='Senior', job_title='AI Engineer', salary=48000)]

In [ ]:
employees_list = [employee.model_dump() for employee in employees]
employees_list

In [31]:
import pandas as pd 

df = pd.DataFrame(employees_list)
df

,name,age,gender,experience_level,job_title,salary
0,Anna Eriksson,28,Female,Entry,Data Scientist,35000
1,Raj Patel,35,Male,Mid level,Data Engineer,40000
2,Sofia Martinez,42,Female,Senior,AI Engineer,48000


In [28]:
df["salary"].mean()

np.float64(41000.0)

In [32]:
df.to_csv("simulated_employees.csv", index=False)